# NexToken Copilot — router QLoRA (2×T4)

Distils the copilot's three router decisions (**plan / route / tool-select**) into a small model.

**Setup (once):**
1. Upload `nextoken-router-ft.zip` as a Kaggle **Dataset** named `nextoken-router-ft` (it holds `ft_router.jsonl`, `qlora_router.py`, `requirements-kaggle.txt`).
2. Attach that dataset to this notebook (Add Input).
3. Settings → Accelerator **GPU T4 ×2**, Internet **On** (for pip + HuggingFace downloads).

Dataset: **212 pairs** — 126 sandbox-labeled from the 37-question battery + 86 route pairs with live qwen2.5:3b phrasings relabeled by the deterministic sandbox router (student routes are too flaky to be labels).

In [ ]:
# Locate the dataset wherever Kaggle mounted it (slug/nesting-proof).
import glob, os
hits = glob.glob("/kaggle/input/**/requirements-kaggle.txt", recursive=True)
assert hits, "Dataset not attached — use 'Add Input' and attach the uploaded dataset."
DATA = os.path.dirname(hits[0])
print("found dataset at:", DATA)
!pip install -q -r {DATA}/requirements-kaggle.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# ~212 pairs × 3 epochs — minutes on a T4. Prints a sanity plan+route generation at the end.
!python {DATA}/qlora_router.py \
    --data {DATA}/ft_router.jsonl \
    --base Qwen/Qwen2.5-1.5B-Instruct \
    --out /kaggle/working/router-lora

In [ ]:
# Quick routing accuracy check on the training battery's route pairs
# (in-sample smoke test, not a real holdout — the gate that matters is `make eval` back home).
import json, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(model, "/kaggle/working/router-lora")

pairs = [json.loads(l) for l in open(f"{DATA}/ft_router.jsonl")]
routes = [p["messages"] for p in pairs if p["messages"][0]["content"].startswith("You are a supervisor")]
ok = 0
for m in routes:
    ids = tok.apply_chat_template(m[:2], add_generation_prompt=True, return_tensors="pt").to(model.device)
    out = model.generate(ids, max_new_tokens=6, do_sample=False, pad_token_id=tok.eos_token_id)
    pred = tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip().split()[:1]
    ok += pred == [m[2]["content"]]
print(f"route accuracy: {ok}/{len(routes)} ({ok/len(routes):.0%})")

In [ ]:
# Package the adapter for download (Output tab → router-lora.zip)
!cd /kaggle/working && zip -qr router-lora.zip router-lora && ls -lh router-lora.zip

**Back home:** unzip into `nextoken-copilot/train/router-lora/`, serve base+adapter (vLLM or Ollama-compatible), point `COPILOT_ROUTER_MODEL` at it, and gate with `make test` + `make eval` (must stay 7/7, 8/8). Optional second run: `--base Qwen/Qwen2.5-3B-Instruct` still fits a T4 in 4-bit.